# Assemble and Check the Filing Summaries

| Item | Description |
|---|---|
| Goal | Put the shard outputs of every T3.3.2 run together into the filing-summary table and check each run. Every filing of the set has one summary, none is empty, all are within the encoder limit, and every record carries the run's prompt version. The quirks seen while the prompt was developed are counted on every run, not enforced. |
| Input | `filing_summaries/<set>/<family>/<version>/shard_<k>.jsonl` and `shard_<k>.json` from T3.3.2 in the T3.3 Interim folder; `filings.parquet` from T3.1.1. |
| Output | `filing_summaries.parquet` in the T3.3 Processed folder: one row per filing, family, and prompt version. Under `R - Results/tables/...`: `table_filing_summary_checks.csv` (one row per shard and run) and `table_example_filing_summaries.csv`. |
| Runs | Four runs are on disk: the three-year set at prompt versions 1.1, 1.2, and 1.3, and the remaining filings at 1.3, all with the `summary_lines_of_business` family. Version 1.3 over both sets is what T3.3.4 and the main measure use; 1.1 and 1.2 stay in the table for the record. |
| Counted, not enforced | Numbered lines per summary and a bare item number at the end (the list versions). Whether a summary refers to its own inputs instead of the company ("the provided summaries do not describe"), the leak that version 1.3 was written to remove. |
| Run | Run 2026-10-31; the table equals the one assembled in October 2026, 22,923 rows. |


## Step 1. Import packages


In [1]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display


## Step 2. Define file locations and settings


In [2]:
PROJECT_FOLDER = Path.cwd().resolve().parents[2]  # <pipeline>/<section>/<subfolder>/ sits three levels below the clone
SCRATCH_FOLDER = Path("/scratch1/acroh/projects/LLM-Text-Analysis-and-Synthesis")  # the one line to change on another machine
NOTEBOOK_SUBFOLDER = Path("T - Three Step Text Analysis Pipeline/T3 - Topic Relevance and Corporate Disclosure/T3.3 - Firm Side - Summaries of the 10-K Item 1 Filings")
NOTEBOOK_NAME = "T3.3.3 - Check - Assemble the Filing Summaries"  # the notebook filename without .ipynb
RAW_FOLDER = SCRATCH_FOLDER / NOTEBOOK_SUBFOLDER / "Raw"
INTERIM_FOLDER = SCRATCH_FOLDER / NOTEBOOK_SUBFOLDER / "Interim"
PROCESSED_FOLDER = SCRATCH_FOLDER / NOTEBOOK_SUBFOLDER / "Processed"
SUMMARIES_FOLDER = INTERIM_FOLDER / "filing_summaries"
FILINGS_FILE = SCRATCH_FOLDER / "T - Three Step Text Analysis Pipeline/T3 - Topic Relevance and Corporate Disclosure/T3.1 - Sample - Firms and 10-K Item 1 Filings" / "Processed" / "filings.parquet"

MAIN_VERSION = "item1-mapreduce-1.3"
MAX_BGE_TOKENS = 512  # the encoder's input limit every summary must respect
N_EXAMPLES = 6  # summaries shown and saved as examples
RANDOM_SEED = 0
FIGURE_FOLDER = PROJECT_FOLDER / "R - Results/figures" / NOTEBOOK_SUBFOLDER / NOTEBOOK_NAME
TABLE_FOLDER = PROJECT_FOLDER / "R - Results/tables" / NOTEBOOK_SUBFOLDER / NOTEBOOK_NAME
MODEL_CACHE = SCRATCH_FOLDER / "cache" / "huggingface"
ENV_FILE = PROJECT_FOLDER / "T - Three Step Text Analysis Pipeline/T3 - Topic Relevance and Corporate Disclosure" / ".env"
for folder in (PROCESSED_FOLDER, TABLE_FOLDER):
    folder.mkdir(parents=True, exist_ok=True)


## Step 3. Define the counts of the quirks

The patterns the smoke runs showed: numbered lists, a list cut at the output cap, and a summary that talks about its inputs.


In [3]:
def count_numbered_lines(text):
    return len(re.findall(r"(?m)^\s*\d+[.)]\s", text))


def has_closing_sentence(text):
    lines = [line.strip() for line in text.strip().splitlines() if line.strip()]
    return bool(lines) and not re.match(r"^\d+[.)]\s", lines[-1])


def ends_truncated(text):
    lines = [line.strip() for line in text.strip().splitlines() if line.strip()]
    return bool(lines) and re.fullmatch(r"\d+[.)]", lines[-1]) is not None


REFERS_TO_INPUT = re.compile(
    r"\b(the|these|provided) (provided |segment |yearly )?summar(y|ies)\b"  # "the provided summaries do not describe"
    r"|\b(this|the) (text|segment)s? (does|do) not\b"  # "this segment does not describe"
    r"|contained within this (text|segment)|not (described|mentioned|provided|contained) in (the|this|these) (text|segment|summar)",
    re.I,
)


def refers_to_input(text):
    return REFERS_TO_INPUT.search(text) is not None


## Step 4. Read every run and check it

For each run folder: all shard files and run records; the filings of its set, none missing and none repeated. No summary is empty or over the limit, and every record carries the prompt version. A shard run in two sessions can hold a filing twice, and the later record wins.


In [4]:
filings = pd.read_parquet(FILINGS_FILE)
expected_by_set = {"three_years": set(filings.loc[filings["in_three_years"].eq(1), "accessionNo"]), "remaining": set(filings.loc[filings["in_three_years"].eq(0), "accessionNo"])}
tables, check_rows = [], []
for run_folder in sorted(SUMMARIES_FOLDER.glob("*/*/*/"), key=lambda p: (p.name, p.parts[-3])):
    set_name, family, version = run_folder.parts[-3], run_folder.parts[-2], run_folder.name
    summary_files = sorted(p for p in run_folder.glob("shard_*.jsonl") if re.fullmatch(r"shard_\d+\.jsonl", p.name))
    rows = []
    for summary_file in summary_files:
        for line in summary_file.read_text(encoding="utf-8").splitlines():
            record = json.loads(line)
            record["shard_id"] = int(summary_file.stem.split("_")[1])
            rows.append(record)
    summaries = pd.DataFrame(rows)
    records = pd.DataFrame([json.loads(p.read_text(encoding="utf-8")) for p in sorted(run_folder.glob("shard_*.json")) if re.fullmatch(r"shard_\d+\.json", p.name)])
    duplicates = int(summaries.duplicated("accessionNo").sum())
    summaries = summaries.drop_duplicates("accessionNo", keep="last")
    missing, extra = expected_by_set[set_name] - set(summaries["accessionNo"]), set(summaries["accessionNo"]) - expected_by_set[set_name]
    assert not missing and not extra, (set_name, version, len(missing), len(extra))
    assert summaries["prompt_version"].eq(version).all() and records["prompt_version"].eq(version).all() and summaries["family"].eq(family).all()
    assert summaries["summary"].str.strip().ne("").all() and summaries["within_bge_limit"].all()
    for name, function in [("n_numbered_lines", count_numbered_lines), ("has_closing_sentence", has_closing_sentence), ("ends_truncated", ends_truncated), ("refers_to_input", refers_to_input)]:
        summaries[name] = summaries["summary"].map(function)
    summaries["set"] = set_name
    tables.append(summaries)
    check = records[["shard_id", "filings_done_this_run", "filings_skipped_as_done", "segments", "model_calls", "gpu", "seconds", "prompt_version"]].copy()
    check.insert(0, "family", family); check.insert(0, "set", set_name)
    check["n_summaries_in_run"], check["duplicates_dropped"] = len(summaries), duplicates
    check["words_median"], check["bge_tokens_max"] = float(summaries["n_words"].median()), int(summaries["n_bge_tokens"].max())
    check["collapsed_filings"], check["re_asked_filings"] = int(summaries["collapse_levels"].gt(0).sum()), int(summaries["reduce_rounds"].gt(1).sum())
    check["numbered_list_share"], check["truncated_share"], check["refers_to_input_share"] = float(summaries["n_numbered_lines"].gt(0).mean()), float(summaries["ends_truncated"].mean()), float(summaries["refers_to_input"].mean())
    check_rows.append(check)
    print(f"{set_name:12s} {version}: {len(summary_files)} shards, {len(summaries):,} summaries ({duplicates} duplicates dropped); words median {int(summaries['n_words'].median())}, max {summaries['n_words'].max()}; "
          f"encoder tokens max {summaries['n_bge_tokens'].max()}; collapsed {int(summaries['collapse_levels'].gt(0).sum())}, re-asked {int(summaries['reduce_rounds'].gt(1).sum())}; "
          f"numbered lists {summaries['n_numbered_lines'].gt(0).mean():.0%}, refers to inputs {summaries['refers_to_input'].mean():.1%}")
checks = pd.concat(check_rows, ignore_index=True)
checks.to_csv(TABLE_FOLDER / "table_filing_summary_checks.csv", index=False)


three_years  item1-mapreduce-1.1: 8 shards, 2,072 summaries (0 duplicates dropped); words median 160, max 398; encoder tokens max 512; collapsed 0, re-asked 5; numbered lists 100%, refers to inputs 37.4%


three_years  item1-mapreduce-1.2: 8 shards, 2,072 summaries (0 duplicates dropped); words median 182, max 380; encoder tokens max 510; collapsed 0, re-asked 8; numbered lists 0%, refers to inputs 50.9%


remaining    item1-mapreduce-1.3: 8 shards, 16,707 summaries (0 duplicates dropped); words median 193, max 412; encoder tokens max 512; collapsed 1, re-asked 94; numbered lists 0%, refers to inputs 1.3%


three_years  item1-mapreduce-1.3: 8 shards, 2,072 summaries (0 duplicates dropped); words median 193, max 396; encoder tokens max 507; collapsed 0, re-asked 9; numbered lists 0%, refers to inputs 1.5%


## Step 5. Build the table

One row per filing, family, and prompt version. The main run, version 1.3 over both sets, must cover every filing of the all-years sample once.


In [5]:
table_columns = ["accessionNo", "gvkey", "conm", "fyear", "set", "family", "prompt_version", "summary", "n_words", "n_bge_tokens", "n_segments", "collapse_levels", "reduce_rounds",
                 "n_numbered_lines", "has_closing_sentence", "ends_truncated", "refers_to_input", "run_started"]
table = pd.concat(tables, ignore_index=True)[table_columns]
assert not table.duplicated(["accessionNo", "family", "prompt_version"]).any()
main = table.loc[table["prompt_version"].eq(MAIN_VERSION)]
assert set(main["accessionNo"]) == set(filings["accessionNo"]) and len(main) == len(filings)
print(f"Table: {len(table):,} rows; main version {MAIN_VERSION}: {len(main):,} filings of {main['gvkey'].nunique():,} firms")
print(table.groupby(["prompt_version", "set"]).size().rename("rows").to_string())


Table: 22,923 rows; main version item1-mapreduce-1.3: 18,779 filings of 1,058 firms
prompt_version       set        
item1-mapreduce-1.1  three_years     2072
item1-mapreduce-1.2  three_years     2072
item1-mapreduce-1.3  remaining      16707
                     three_years     2072


## Step 6. Show examples from the main run

Random summaries, with the shortest and the longest, so the reader sees the range.


In [6]:
rng = np.random.default_rng(RANDOM_SEED)
chosen = pd.concat([main.iloc[rng.choice(len(main), size=N_EXAMPLES - 2, replace=False)], main.nsmallest(1, "n_words"), main.nlargest(1, "n_words")])
examples = chosen[["accessionNo", "conm", "fyear", "n_segments", "n_words", "n_bge_tokens", "refers_to_input", "summary"]]
examples.to_csv(TABLE_FOLDER / "table_example_filing_summaries.csv", index=False)
for example in examples.itertuples(index=False):
    print(f"=== {example.conm} {example.fyear} | {example.n_segments} segments | {example.n_words} words ===\n{example.summary}\n")


=== BANK OF AMERICA CORP 2011 | 2 segments | 114 words ===
Bank of America operates as a financial holding company providing a diversified range of banking, investing, asset management, and risk management products and services. The company serves individual consumers, small- and middle-market businesses, institutional investors, large corporations, and governments through six specific business segments: Deposits, Card Services, Consumer Real Estate Services, Global Commercial Banking, Global Banking & Markets, and Global Wealth & Investment Management, with remaining operations recorded in All Other. These segments deliver services through banking and various nonbanking subsidiaries located throughout the United States and in international markets. The company's principal executive offices are located in Charlotte, North Carolina, and its website provides access to its annual reports and corporate governance materials.

=== EOG RESOURCES INC 2004 | 4 segments | 241 words ===
EOG Resou

## Step 7. Save the table


In [7]:
table.to_parquet(PROCESSED_FOLDER / "filing_summaries.parquet", index=False)
print(f"Saved filing_summaries.parquet ({len(table):,} rows), table_filing_summary_checks.csv ({len(checks)} rows), and table_example_filing_summaries.csv")


Saved filing_summaries.parquet (22,923 rows), table_filing_summary_checks.csv (32 rows), and table_example_filing_summaries.csv
